# Day 06 — Async Context Managers

**Phase 1 · Module 1: Plan-Execute** · ~30 min

### What I practice today
- [ ] Understand `with` / `async with` — **guaranteed setup + teardown**
- [ ] Write an **async context manager** two ways: a class (`__aenter__`/`__aexit__`) and `@asynccontextmanager`
- [ ] Wrap an **LLM client lifecycle** so the connection *always* closes — even on error

> Feeds today's LangGraph notebook: the Plan-and-Execute agent fires **many** LLM calls (plan → execute → replan). Each needs a client that opens once, is reused, and is **guaranteed to close**. An async context manager is how you get that guarantee.

## 1. The problem: setup you must *always* pair with teardown

Some resources must be **released** no matter what — a file, a socket, a DB connection, an HTTP session to an LLM API. If your code raises halfway through, a forgotten `close()` leaks the resource.

The naive fix is `try/finally`. It works but it's noisy and easy to forget:

In [1]:
# Without a context manager: you MUST remember try/finally, every time.
f = open('/tmp/note.txt', 'w')
try:
    f.write('hello')
    # imagine an exception here...
finally:
    f.close()          # teardown you must never forget
    print('file closed:', f.closed)

file closed: True


The `with` statement bottles that `try/finally` into one line. Setup on enter, teardown on exit — **automatic**, even if the body raises:

In [2]:
# WITH a context manager: teardown is automatic and un-forgettable.
with open('/tmp/note.txt', 'w') as f:
    f.write('hello')
# f is already closed here - the 'with' block guaranteed it.
print('file closed:', f.closed)

file closed: True


☝️ Same guarantee, no `finally` to forget. **That's the whole point of a context manager: pair every setup with its teardown, automatically.**

## 2. Build your own (sync) — the `__enter__` / `__exit__` protocol

Any object with **`__enter__`** (runs on the way in) and **`__exit__`** (runs on the way out) works with `with`. Here's a fake DB connection so you can *see* both halves fire:

In [3]:
class DBConnection:
    """A pretend connection so we can watch setup and teardown fire."""
    def __enter__(self):
        print('  __enter__  -> opening connection')
        self.open = True
        return self                # whatever 'as x' binds to

    def __exit__(self, exc_type, exc, tb):
        print('  __exit__   -> closing connection')
        self.open = False
        return False               # False = don't swallow exceptions

with DBConnection() as conn:
    print('  inside     -> conn.open =', conn.open)
print('after with -> conn.open =', conn.open)

  __enter__  -> opening connection
  inside     -> conn.open = True
  __exit__   -> closing connection
after with -> conn.open = False


☝️ `__exit__` receives the exception info (`exc_type, exc, tb`). Returning **`False`** means *"I cleaned up, but let the error keep propagating"* — you almost always want that. Returning `True` would **swallow** the exception (rarely what you want).

## 3. The shortcut — `@contextmanager` (no class needed)

Writing a whole class for two methods is heavy. `contextlib.contextmanager` turns a **generator** into a context manager: everything before `yield` is `__enter__`, everything after is `__exit__`.

In [4]:
from contextlib import contextmanager

@contextmanager
def db_connection():
    print('  setup    -> open')          # __enter__
    conn = {'open': True}
    try:
        yield conn                        # hand resource to the 'with' body
    finally:
        conn['open'] = False              # __exit__ - runs no matter what
        print('  teardown -> close')

with db_connection() as conn:
    print('  inside   ->', conn)
print('closed  ->', conn)

  setup    -> open
  inside   -> {'open': True}
  teardown -> close
closed  -> {'open': False}


☝️ The `finally` around the `yield` is what makes teardown bulletproof — if the body raises, Python throws the exception *into* the generator at the `yield`, and `finally` still runs the cleanup.

## 4. Why *async*? Because opening/closing can itself `await`

An LLM client, an HTTP session, or a connection pool opens over the **network** — that's I/O, and I/O should be `await`ed so the event loop isn't blocked (Day 03). But a plain `__enter__` **cannot `await`** — it's a normal function.

So Python has an async twin:

| Sync | Async |
|------|-------|
| `with` | `async with` |
| `__enter__` | `async __aenter__` (can `await`) |
| `__exit__` | `async __aexit__` (can `await`) |
| `@contextmanager` | `@asynccontextmanager` |

Use the async version whenever **opening or closing the resource is itself an `await`** — which is exactly the case for a network LLM client.

## 5. Async context manager, the class way — an LLM client lifecycle

Model the thing today's LangGraph agent actually needs: a client that **connects** (await), gets **reused** for many calls, then **disconnects** (await). `asyncio.sleep` stands in for real network latency so this runs with no API key.

In [5]:
import asyncio

class LLMClient:
    """Async lifecycle for a network LLM client: connect -> use -> close."""
    def __init__(self, model: str):
        self.model = model
        self.connected = False

    async def __aenter__(self):
        await asyncio.sleep(0.05)          # opening a real session is I/O
        self.connected = True
        print(f'  __aenter__ -> connected to {self.model}')
        return self

    async def __aexit__(self, exc_type, exc, tb):
        await asyncio.sleep(0.05)          # closing the session is I/O too
        self.connected = False
        print('  __aexit__  -> session closed')
        return False

    async def complete(self, prompt: str) -> str:
        assert self.connected, 'client not connected!'
        await asyncio.sleep(0.02)          # the actual model call
        return f'[{self.model}] answered: {prompt[:30]}...'

async def main():
    async with LLMClient('gemini-2.5-flash') as llm:      # note: async with
        print(' ', await llm.complete('Summarise the credit policy doc'))
        print(' ', await llm.complete('Summarise the fraud policy doc'))
    print('connected after block?', llm.connected)

await main()

  __aenter__ -> connected to gemini-2.5-flash
  [gemini-2.5-flash] answered: Summarise the credit policy do...
  [gemini-2.5-flash] answered: Summarise the fraud policy doc...
  __aexit__  -> session closed
connected after block? False


☝️ **One** connect, **two** calls reusing it, **one** guaranteed close — the pattern a multi-call agent lives on. Opening 3 LLM calls with 3 separate connects/closes would be slow and wasteful; the context manager opens once and cleans up once.

## 6. The async shortcut — `@asynccontextmanager`

Same as §3, but async: an **async generator** with one `yield`. Less code than the class, and the `finally` still guarantees teardown.

In [6]:
from contextlib import asynccontextmanager

@asynccontextmanager
async def llm_client(model: str):
    await asyncio.sleep(0.05)              # __aenter__
    client = LLMClient.__new__(LLMClient)
    client.model, client.connected = model, True
    print(f'  open  -> {model}')
    try:
        yield client
    finally:
        client.connected = False          # __aexit__ - always runs
        await asyncio.sleep(0.05)
        print('  close -> done')

async def main():
    async with llm_client('gemini-2.5-flash') as llm:
        print(' ', await llm.complete('plan the summarisation task'))

await main()

  open  -> gemini-2.5-flash
  [gemini-2.5-flash] answered: plan the summarisation task...
  close -> done


☝️ For simple open/close, `@asynccontextmanager` is the idiomatic choice — reach for the class only when the resource needs to hold real methods/state (like our `LLMClient` with `.complete`).

## 7. The payoff: teardown runs **even when the body crashes**

This is why we bother. If an LLM call blows up mid-agent, you still **must** close the session or you leak connections. `__aexit__` / the `finally` runs on the way out regardless — success *or* exception.

In [7]:
async def main():
    try:
        async with LLMClient('gemini-2.5-flash') as llm:
            print('  calling model...')
            raise RuntimeError('model API returned 500')   # something explodes
    except RuntimeError as e:
        print('  caught:', e)
    print('connected after crash?', llm.connected, '  <- still closed cleanly')

await main()

  __aenter__ -> connected to gemini-2.5-flash
  calling model...
  __aexit__  -> session closed
  caught: model API returned 500
connected after crash? False   <- still closed cleanly


☝️ The `RuntimeError` propagated (we caught it *outside*), but `__aexit__` **still closed the session first**. No leaked connection. This is exactly the safety a plan-execute agent needs when a step fails and triggers replanning.

## 8. Tie-in — one client, wrapped once, used by every node

In today's LangGraph Plan-and-Execute agent, the **planner**, **executor**, and **replan** nodes all call the model. Open the client **once** in an `async with`, run the whole graph inside it, and the session closes when the graph finishes — even if a node raises.

```mermaid
flowchart TD
    subgraph CM["async with LLMClient(...)  — one session"]
        direction LR
        P[planner] --> E[executor] --> R[replan]
    end
    OPEN([__aenter__: connect]) --> CM --> CLOSE([__aexit__: close])
```

In [8]:
async def run_agent(goal: str):
    async with LLMClient('gemini-2.5-flash') as llm:      # opened ONCE
        plan   = await llm.complete(f'PLAN: {goal}')      # planner node
        step1  = await llm.complete('EXECUTE: step 1')    # executor node
        step2  = await llm.complete('EXECUTE: step 2')    # executor node
        replan = await llm.complete('REPLAN: done?')      # replan node
        return [plan, step1, step2, replan]
    # <- session guaranteed closed here, whatever happened above

for line in await run_agent('summarise 3 policy documents'):
    print(' ', line)

  __aenter__ -> connected to gemini-2.5-flash
  __aexit__  -> session closed
  [gemini-2.5-flash] answered: PLAN: summarise 3 policy docum...
  [gemini-2.5-flash] answered: EXECUTE: step 1...
  [gemini-2.5-flash] answered: EXECUTE: step 2...
  [gemini-2.5-flash] answered: REPLAN: done?...


☝️ Four model calls, **one** connection lifecycle. Swap `LLMClient` for a real `ChatGoogleGenerativeAI` session and the shape is identical — that's the bridge to the LangGraph notebook.

## 9. Recap + checklist

| Tool | Why it was used here |
|------|----------------------|
| **`with` / `async with`** | pair every setup with its teardown — automatically, un-forgettably |
| **`__enter__` / `__exit__`** | sync context-manager protocol (setup in, teardown out) |
| **`__aenter__` / `__aexit__`** | async twin — used when open/close is itself an `await` (network I/O) |
| **`@contextmanager` / `@asynccontextmanager`** | one generator with a `yield` instead of a whole class |
| **`finally` around `yield`** | guarantees teardown even when the body raises |
| **`__aexit__` return `False`** | clean up *and* let the real error propagate |
| **one `async with` around the graph** | a single LLM session shared by planner/executor/replan nodes |

- [x] `async with` = guaranteed async setup + teardown
- [x] Wrote an async context manager as a class **and** with `@asynccontextmanager`
- [x] Proved teardown fires on error → no leaked LLM connections